# Theory runner — generic compute_cumulants front-end

Loads any saved theory file from `theories/*.theory.py` and runs the
full pipeline (mean-field solve → diagram enumeration → Phase J
integration) against it.  One notebook works for every theory you
build.

**Workflow**:

1. Set `THEORY_NAME` in the configuration cell (matches a filename in
   `theories/` — without the `.theory.py` suffix).
2. Optionally override `fundamental` (numerical parameter values), `k`,
   `max_ell`, `external_fields`, etc.  Defaults come from the
   theory file's `DEFAULT_FUNDAMENTAL` and `METADATA`.
3. Run all cells.

The companion notebook `theory_builder.ipynb` (UI for **creating**
new theory files) will save into the same `theories/` directory so
they're immediately loadable here.


## 1. Setup

In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, time, importlib, importlib.util
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from pipeline import compute_cumulants, save_npz, save_csv, generate_report


## 2. Configuration

`THEORY_NAME` must match a file `theories/<name>.theory.py`.  The
loader below walks `theories/` and lists what's available.


In [ ]:
THEORIES_DIR = os.path.abspath('../theories')

# Discover saved theory files
def list_theories():
    files = sorted(f for f in os.listdir(THEORIES_DIR)
                   if f.endswith('.theory.py'))
    return [f[:-len('.theory.py')] for f in files]

print('Available theories in', THEORIES_DIR)
for name in list_theories():
    print(f'  - {name}')


In [ ]:
# ─── Pick which theory to run ───────────────────────────────────
THEORY_NAME = 'quadratic_hawkes'

# ─── Compute settings ──────────────────────────────────────────
# (Overrides the theory file's METADATA defaults; set to None to
#  use the file's recommendations.)
K               = None     # 1, 2, 3, ...     None ⇒ METADATA['k_default']
MAX_ELL         = None     # 0 (tree), 1 (1-loop), ...
EXTERNAL_FIELDS = None     # [('n', 1), ('n', 2)] etc.; None ⇒ METADATA
TAU_MAX         = None
TAU_STEP        = None

# ─── Numerical parameter values ────────────────────────────────
# None ⇒ use the theory file's DEFAULT_FUNDAMENTAL.  Otherwise
# provide a dict with the same keys.
FUNDAMENTAL_OVERRIDE = None

# ─── MF solver (DAE multi-root) ────────────────────────────────
# When the theory declares ``.equation(...)`` calls, the DAE
# solver runs.  fixed_point_index picks WHICH sorted root the
# diagrammatic expansion uses (0 = lowest, 1 = next, ...).  None
# ⇒ METADATA['fixed_point_index_default'] (or 0).
FIXED_POINT_INDEX = None
# Optional per-variable initial-guess box for multi-start Newton.
# None ⇒ METADATA['seed_box_default'] (or domain-aware default).
MF_DAE_SEED_BOX   = None

# ─── Pipeline parallelism ──────────────────────────────────────
PARALLEL  = True
N_WORKERS = None

# ─── Output ────────────────────────────────────────────────────
SAVE_NPZ = True
SAVE_CSV = True
SAVE_PDF = True

# ─── Spatial models (Laplacian theories) ───────────────────────
# A spatial theory needs a grid of x points for the real-space
# correlator C(x, τ).  None ⇒ METADATA['spatial_grid'], else a
# default linspace.  Ignored for non-spatial (temporal) theories.
SPATIAL_GRID = None        # e.g. list(np.linspace(0.0, 6.0, 25))


## 3. Load the theory file

In [ ]:
def load_theory(name):
    """Import theories/<name>.theory.py as a module."""
    path = os.path.join(THEORIES_DIR, f'{name}.theory.py')
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f'No theory file at {path}.  Available: {list_theories()}')
    spec = importlib.util.spec_from_file_location(
        f'theories.{name}', path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


theory_mod = load_theory(THEORY_NAME)

# Build the model dict
model = theory_mod.build()
print(f'Loaded theory: {model["name"]!r}')

# Resolve compute settings (override → metadata → fallback)
meta = getattr(theory_mod, 'METADATA', {}) or {}
fundamental = (FUNDAMENTAL_OVERRIDE if FUNDAMENTAL_OVERRIDE is not None
               else theory_mod.DEFAULT_FUNDAMENTAL)
k               = K if K is not None else meta.get('k_default', 2)
max_ell         = MAX_ELL if MAX_ELL is not None else meta.get('ell_default', 0)
external_fields = (EXTERNAL_FIELDS if EXTERNAL_FIELDS is not None
                   else meta.get('recommended_external_fields',
                                 [('n', 1)] * k))
tau_max         = TAU_MAX  if TAU_MAX  is not None else meta.get('tau_max', 50.0)
fixed_point_index = (FIXED_POINT_INDEX if FIXED_POINT_INDEX is not None
                     else int(meta.get('fixed_point_index_default', 0)))
mf_dae_seed_box   = (MF_DAE_SEED_BOX if MF_DAE_SEED_BOX is not None
                     else meta.get('seed_box_default'))
tau_step        = TAU_STEP if TAU_STEP is not None else meta.get('tau_step', 0.5)

# Spatial models: resolve an x-grid and clamp k / max_ell to the v1 scope
# (k=2 two-point, max_ell<=2).  Non-spatial theories leave spatial_grid=None.
is_spatial = bool(model.get('spatial'))
if is_spatial:
    import numpy as _np
    spatial_grid = SPATIAL_GRID if SPATIAL_GRID is not None else meta.get('spatial_grid')
    if spatial_grid is None:
        spatial_grid = list(_np.linspace(0.0, 6.0, 25))
        print("  (no METADATA['spatial_grid']; using default linspace(0, 6, 25))")
    spatial_grid = _np.asarray(spatial_grid, dtype=float)
    if k != 2:
        print(f'  [spatial] k={k} -> forcing k=2 (v1 supports two-point only)'); k = 2
    if max_ell > 2:
        print(f'  [spatial] max_ell={max_ell} -> capping at 2 (v1 limit)'); max_ell = 2
    _fn = model['physical_fields'][0]['name']
    if external_fields is None or len(external_fields) != 2:
        external_fields = [(_fn, 1), (_fn, 1)]
else:
    spatial_grid = None

print(f'\nFields:     {[f["name"] for f in model["physical_fields"]]}')
print(f'Parameters: {[p["name"] for p in model["parameters"]]}')
print(f'\nCompute settings:')
print(f'  k               = {k}')
print(f'  max_ell         = {max_ell}')
print(f'  external_fields = {external_fields}')
print(f'  tau range       = [-{tau_max}, {tau_max}], step {tau_step}')
print(f'  spatial         = {is_spatial}' + (f' ({len(spatial_grid)} x-points)' if is_spatial else ''))
print(f'\nFundamental parameter values:')
for pname, pval in fundamental.items():
    print(f'  {pname:8} = {pval}')


## 4. Run the pipeline

Calls `compute_cumulants` with the loaded model + resolved settings.
This runs:
- FieldTheory expansion
- Symbolic propagator construction (cached)
- Numerical pole + residue computation
- Mean-field saddle solve
- Diagram enumeration → typing → causal filter → dedup (cached)
- Phase J time-domain integration on the τ grid


In [ ]:
t0 = time.perf_counter()
result = compute_cumulants(
    model           = model,
    k               = k,
    max_ell         = max_ell,
    fundamental     = fundamental,
    external_fields = external_fields,
    spatial_grid    = spatial_grid,
    tau_max         = tau_max,
    tau_step        = tau_step,
    parallel        = PARALLEL,
    n_workers       = N_WORKERS,
    fixed_point_index = fixed_point_index,
    mf_dae_seed_box   = mf_dae_seed_box,
    verbose         = True,
)
print(f'\nFinished in {time.perf_counter() - t0:.1f}s')


## 5. Inspect the result

The result dict exposes:
- `mf` — :class:`pipeline.access.MeanField` accessor (`mf['n', 1]`, etc.)
- `params` — :class:`pipeline.access.Parameters` accessor
- `C_tau` — full sum across loop orders
- `C_tau_by_ell` — per-loop-order decomposition (dict keyed by `ell`)
- `tau_grid` — τ values
- `diagrams` — list of typed diagrams with prefactors and ell tags


In [ ]:
mf     = result['mf']
params = result['params']

print('Mean-field saddle values:')
for name in mf:
    vals = mf[name]
    print(f'  {name!r:6} = {vals}')

_diags = result.get('diagrams')
if _diags is not None:
    print('\nDiagram count by loop order:')
    from collections import Counter
    ell_counter = Counter(d['ell'] for d in _diags)
    for ell, count in sorted(ell_counter.items()):
        print(f'  ell={ell}: {count} diagrams')
    print(f'  total:  {len(_diags)}')
else:  # spatial result: diagrams summed inside the integrator
    _si = result.get('spatial_info', {}) or {}
    print('\n[spatial] integrated {} live diagram(s); max_ell={}'.format(
          _si.get('n_live_diagrams', '?'), _si.get('max_ell', '?')))

print(f'\ntau_grid: {len(result["tau_grid"])} points')
print(f'C_tau shape: {result["C_tau"].shape if result["C_tau"] is not None else None}')


### C(τ) curve(s)

For k=1 the cumulant is a scalar (firing rate) — plot is trivial.
For k=2 the cumulant is a function of τ — plot tree, each loop
order, and the total.
For k≥3 you'd need a different slice; this notebook leaves
evaluation to the caller.


In [ ]:
if is_spatial:
    import numpy as _np
    C   = _np.real(result['C_tau_x']); xs = _np.asarray(result['spatial_grid'])
    tau = _np.asarray(result['tau_grid']); i0 = int(_np.argmin(_np.abs(tau)))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].plot(xs, C[i0], 'o-', color='#27AE60')
    ax[0].set_xlabel('x'); ax[0].set_ylabel('C(x, 0)')
    ax[0].set_title(f'{model["name"]}: equal-time C(x, 0)'); ax[0].grid(alpha=0.25)
    if len(tau) > 1:
        im = ax[1].imshow(C, aspect='auto', origin='lower',
                          extent=[xs.min(), xs.max(), tau.min(), tau.max()])
        ax[1].set_xlabel('x'); ax[1].set_ylabel(r'$\tau$')
        ax[1].set_title(r'$C(x, \tau)$'); fig.colorbar(im, ax=ax[1])
    else:
        ax[1].axis('off')
    plt.tight_layout(); plt.show()
elif result['C_tau'] is None:
    print('No 1-D τ slice for k≥3; use result["total_C_batch"] manually.')
elif k == 1:
    val = result['C_tau'].real.mean()
    print(f'k=1 result (constant on τ grid): {val:.6e}')
    print('(For stationary systems C^(1) is τ-independent — value above '
          'is the rate or rate + loop correction.)')
else:    # k == 2
    tau   = result['tau_grid']
    fig, ax = plt.subplots(figsize=(10, 4))

    # Per-loop-order curves
    if max_ell > 0:
        colors = ['#3F00FF', '#E74C3C', '#27AE60', '#F39C12']
        for ell, C_ell in result['C_tau_by_ell'].items():
            if C_ell is None:
                continue
            label = (f'tree (ell=0)' if ell == 0 else f'{ell}-loop')
            ls = '--' if ell == 0 else '-'
            ax.plot(tau, C_ell.real, linewidth=1.5, linestyle=ls,
                    color=colors[ell % len(colors)], label=label)
        # Total
        ax.plot(tau, result['C_tau'].real, color='black', linewidth=2.0,
                label='total', alpha=0.7)
    else:
        ax.plot(tau, result['C_tau'].real, color='#3F00FF', linewidth=1.5,
                label='tree')

    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel(r'$\tau$')
    ax.set_ylabel(r'$C^{(2)}$')
    ax.set_title(f'{model["name"]}: C(τ) decomposition')
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.show()


## 6. Save outputs

In [ ]:
out_dir = os.path.abspath(f'../pipeline_outputs/{THEORY_NAME}')
os.makedirs(out_dir, exist_ok=True)
slug = f'{THEORY_NAME}_k{k}_ell{max_ell}'

if SAVE_NPZ:
    save_npz(result, f'{out_dir}/{slug}.npz')
    print(f'Saved NPZ: {out_dir}/{slug}.npz')

if SAVE_CSV:
    save_csv(result, f'{out_dir}/{slug}.csv')
    print(f'Saved CSV: {out_dir}/{slug}.csv')

if SAVE_PDF:
    generate_report(
        model           = model,
        k               = k,
        fundamental     = fundamental,
        external_fields = external_fields,
        output_pdf      = f'{out_dir}/{slug}_report.pdf',
        result          = result,
        verbose         = False,
    )
    print(f'Saved PDF: {out_dir}/{slug}_report.pdf')


---

### Want a different theory?

Edit `THEORY_NAME` in the configuration cell and re-run all cells.
The currently-saved theories are listed in section 2 — that list
will grow automatically as you save new ones from the
`theory_builder.ipynb` UI.

To override numerical parameters without editing the theory file
itself, set `FUNDAMENTAL_OVERRIDE` in the configuration cell, e.g.:

```python
FUNDAMENTAL_OVERRIDE = {
    'E':     [0.5, 0.5],     # different drive
    'w':     [[0.3, 0.25], [0.3, 0.35]],
    'tau':   10.0,
    'a':     0.44,
    'tau_g': 5.0,
}
```
